# 5.3 硬件感知优化

LLM推理的瓶颈取决于推理阶段：prefill阶段是计算受限（compute-bound），decode阶段是内存带宽受限（memory-bound）。硬件感知优化针对不同瓶颈采取不同策略。

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

### 内存带宽 vs 计算瓶颈分析

In [ ]:
class RooflineAnalyzer:
    """Roofline模型分析器"""
    def __init__(self, peak_flops=10e12, peak_bandwidth=100e9):
        self.peak_flops = peak_flops
        self.peak_bandwidth = peak_bandwidth
        self.arithmetic_intensity_threshold = peak_flops / peak_bandwidth

    def analyze_operation(self, name, flops, bytes_accessed):
        """分析单个操作的瓶颈"""
        arithmetic_intensity = flops / max(bytes_accessed, 1)
        is_compute_bound = arithmetic_intensity > self.arithmetic_intensity_threshold
        max_throughput = min(self.peak_flops, self.peak_bandwidth * arithmetic_intensity)
        return {
            'name': name,
            'flops': flops,
            'bytes': bytes_accessed,
            'arithmetic_intensity': arithmetic_intensity,
            'is_compute_bound': is_compute_bound,
            'max_throughput_gflops': max_throughput / 1e9,
        }

    def analyze_llm_inference(self, hidden_dim, n_layers, seq_len, n_heads, batch_size=1, dtype_bytes=2):
        """分析LLM推理的计算特征"""
        results = {}

        prefill_flops = 2 * batch_size * seq_len * hidden_dim * hidden_dim * n_layers * 3
        prefill_bytes = (hidden_dim * hidden_dim * 3 * n_layers * dtype_bytes +
                         batch_size * seq_len * hidden_dim * dtype_bytes)
        results['prefill'] = self.analyze_operation(
            'Prefill', prefill_flops, prefill_bytes)

        decode_flops = 2 * batch_size * 1 * hidden_dim * hidden_dim * n_layers * 3
        decode_bytes = (hidden_dim * hidden_dim * 3 * n_layers * dtype_bytes +
                        batch_size * seq_len * hidden_dim * dtype_bytes * 2 +
                        batch_size * 1 * hidden_dim * dtype_bytes)
        results['decode'] = self.analyze_operation(
            'Decode', decode_flops, decode_bytes)

        return results

analyzer = RooflineAnalyzer(peak_flops=10e12, peak_bandwidth=100e9)

configs = [
    ('7B FP16', 4096, 32, 512, 32),
    ('7B INT4', 4096, 32, 512, 32),
    ('1.5B FP16', 2048, 24, 512, 16),
    ('1.5B INT4', 2048, 24, 512, 16),
]

print(f"=== LLM推理瓶颈分析 (Roofline模型) ===")
print(f"假设: 峰值算力=10 TFLOPS, 峰值带宽=100 GB/s")
print(f"算术强度阈值: {analyzer.arithmetic_intensity_threshold:.1f} FLOP/Byte")

for name, dim, layers, seq_len, heads in configs:
    dtype_bytes = 2 if 'FP16' in name else 0.5
    results = analyzer.analyze_llm_inference(dim, layers, seq_len, heads, dtype_bytes=dtype_bytes)
    print(f"\n--- {name} ---")
    for phase, info in results.items():
        bottleneck = '计算受限' if info['is_compute_bound'] else '带宽受限'
        print(f"  {phase}: AI={info['arithmetic_intensity']:.2f} FLOP/B, {bottleneck}")

print(f"\n=== 关键洞察 ===")
print(f"1. Prefill阶段: 通常是计算受限 → 优化方向: 算子融合/并行计算")
print(f"2. Decode阶段: 通常是带宽受限 → 优化方向: 量化(减少数据搬运)/权重预取")
print(f"3. 量化对decode加速最显著: 减少权重读取量")

### 功耗预算优化

In [ ]:
class PowerBudgetOptimizer:
    """功耗预算优化器"""
    def __init__(self, tdp_watts=5.0):
        self.tdp = tdp_watts
        self.strategies = {
            'DVFS': {
                'description': '动态电压频率调整',
                'power_saving': '20-40%',
                'performance_impact': '10-30%减速',
            },
            '精度自适应': {
                'description': '根据负载动态切换计算精度',
                'power_saving': '30-50%',
                'performance_impact': '精度轻微下降',
            },
            '推理调度': {
                'description': '错峰执行推理任务，避免持续高负载',
                'power_saving': '15-25%',
                'performance_impact': '延迟增加',
            },
            '模型切换': {
                'description': '简单任务用小模型，复杂任务用大模型',
                'power_saving': '40-60%',
                'performance_impact': '简单任务精度略降',
            },
        }

    def recommend(self, scenario):
        """根据场景推荐功耗优化策略"""
        if scenario == 'realtime_chat':
            return ['DVFS', '精度自适应']
        elif scenario == 'background':
            return ['推理调度', '模型切换']
        elif scenario == 'battery_low':
            return ['模型切换', 'DVFS', '推理调度']
        return list(self.strategies.keys())

optimizer = PowerBudgetOptimizer(tdp_watts=5.0)
print(f"=== 功耗预算优化 (TDP={optimizer.tdp}W) ===")

for name, info in optimizer.strategies.items():
    print(f"\n{name}:")
    print(f"  描述: {info['description']}")
    print(f"  节能: {info['power_saving']}")
    print(f"  影响: {info['performance_impact']}")

print(f"\n--- 场景推荐 ---")
for scenario in ['realtime_chat', 'background', 'battery_low']:
    recs = optimizer.recommend(scenario)
    print(f"  {scenario}: {recs}")

print(f"\n端侧功耗约束是硬约束，必须在TDP内完成推理")
print(f"关键: 在性能、精度、功耗三角中找到平衡点")